<div style="background: linear-gradient(135deg, #1F3864 0%, #2E5FA3 100%); padding: 48px 40px; border-radius: 12px; margin-bottom: 8px;">
  <h1 style="color: white; font-size: 2.4em; font-weight: 800; margin: 0 0 8px 0; letter-spacing: -0.5px;">Deep Learning for Business Analytics</h1>
  <h2 style="color: #a8c4e8; font-size: 1.3em; font-weight: 400; margin: 0 0 16px 0; font-style: italic;">From Basics to Large Language Models</h2>
  <p style="color: #c8ddf5; font-size: 0.95em; margin: 0 0 24px 0;">Dr. M. Ramasubramaniam &amp; Mr. Daniel Peter</p>
  <div style="background: rgba(255,255,255,0.12); border-radius: 8px; padding: 16px 20px; display: inline-block;">
    <span style="color: #ddeeff; font-size: 1.05em; font-weight: 600;">&#128214; Chapter 4 &nbsp;&middot;&nbsp; Feedforward Neural Networks</span>
  </div>
</div>
<div style="background: #f0f4fa; border-left: 5px solid #2E5FA3; padding: 14px 20px; border-radius: 0 8px 8px 0; margin-top: 4px; color: #333; font-size: 0.97em;">
  <em>Building, training, and evaluating feedforward neural networks in PyTorch &mdash; from digit recognition to predicting customer churn.</em>
</div>

## What This Chapter Covers

| Section | Topics |
|---------|--------|
| 4.1 FFN Architecture | Layers and neurons · `nn.Module` · Counting parameters · Design decisions |
| 4.2 Warm-Up: MNIST Digit Recognition | Loading data · Training with shared helpers · Confusion matrix |
| 4.3 Evaluation Metrics for Classification | Accuracy · Precision · Recall · F1 · When accuracy is not enough |
| 4.4 Business Application: Customer Churn | EDA · Class imbalance · FFN classifier · Full evaluation suite |
| 4.5 Pipeline: Saving and Deploying the Churn Model | `ModelPipeline` from Chapter 3 · Save · Load · Predict · Retrain |

> **Datasets used in this chapter:**
> - **MNIST** — loaded directly via `torchvision.datasets` (no download required)
> - **Telco Customer Churn** — downloaded from Kaggle in Section 4.4

---
## Setup

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Chapter 4 — Setup
# Run this cell first. Expected time: under 2 minutes on Colab.
# ─────────────────────────────────────────────────────────────────────────────

!pip install --quiet torch torchvision numpy pandas matplotlib seaborn scikit-learn kaggle tqdm dill

import copy, datetime, os, json, getpass
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

np.random.seed(42)
torch.manual_seed(42)

device = (
    torch.device('cuda') if torch.cuda.is_available() else
    torch.device('mps')  if hasattr(torch.backends, 'mps') and
                             torch.backends.mps.is_available() else
    torch.device('cpu')
)

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor']   = '#f8f9fa'
plt.rcParams['axes.grid']        = True
plt.rcParams['grid.alpha']       = 0.4
plt.rcParams['font.size']        = 11

print(f'Setup complete. Running on: {device}')

---
## Reusable Training Helpers

Chapter 3 established three standalone training functions that keep the model class clean and eliminate duplicated loop code. We use the **same helpers here unchanged** — defined once, called for both MNIST and the churn model with only the parameters changing between calls.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Training helpers — identical to Chapter 3
# Defined once; used for MNIST (§4.2) and churn (§4.4).
# ─────────────────────────────────────────────────────────────────────────────

def train_one_epoch(model, loader, criterion, optimiser, device):
    """One full pass over the training data. Returns mean loss."""
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimiser.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimiser.step()
        total_loss += loss.item() * len(X_batch)
    return total_loss / len(loader.dataset)


def evaluate(model, loader, criterion, device):
    """Evaluate on a DataLoader without updating weights. Returns mean loss."""
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            total_loss += criterion(model(X_batch), y_batch).item() * len(X_batch)
    return total_loss / len(loader.dataset)


def run_training(model, train_loader, val_loader, criterion, optimiser,
                 num_epochs, device, scheduler=None, val_loss_scheduler=None,
                 early_stopping=None, verbose_every=10):
    """Train for num_epochs. Returns (train_losses, val_losses).

    scheduler          : epoch-based scheduler (StepLR, CosineAnnealingLR).
                         Called as scheduler.step() after each epoch.
    val_loss_scheduler : loss-based scheduler (ReduceLROnPlateau).
                         Called as val_loss_scheduler.step(val_loss) after each epoch.
    early_stopping     : EarlyStopping instance. Training stops when it returns True.
    """
    train_losses, val_losses = [], []
    for epoch in range(num_epochs):
        tl = train_one_epoch(model, train_loader, criterion, optimiser, device)
        vl = evaluate(model, val_loader, criterion, device)
        train_losses.append(tl)
        val_losses.append(vl)
        if scheduler:
            scheduler.step()
        if val_loss_scheduler:
            val_loss_scheduler.step(vl)      # ReduceLROnPlateau needs val_loss
        if verbose_every and (epoch % verbose_every == 0 or epoch == num_epochs - 1):
            print(f'  Epoch {epoch:3d}: train={tl:.4f}  val={vl:.4f}')
        if early_stopping and early_stopping.step(vl, model):
            print(f'  Early stopping at epoch {epoch}  (best val: {early_stopping.best_loss:.4f})')
            early_stopping.restore_best(model)
            break
    return train_losses, val_losses


class EarlyStopping:
    """Halt training when validation loss stops improving.
    Identical to Chapter 3 — copied here so this notebook is self-contained.
    """
    def __init__(self, patience=8, min_delta=1e-4):
        self.patience     = patience
        self.min_delta    = min_delta
        self.best_loss    = float('inf')
        self.counter      = 0
        self.best_weights = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss    = val_loss
            self.counter      = 0
            self.best_weights = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore_best(self, model):
        if self.best_weights is not None:
            model.load_state_dict(self.best_weights)


print('Helpers defined: train_one_epoch | evaluate | run_training | EarlyStopping')

---
# 4.1 FFN Architecture

## What a feedforward network actually does

A **feedforward neural network (FFN)** — sometimes called a multi-layer perceptron (MLP) — is the foundational architecture in deep learning. Every more specialised architecture in later chapters (CNNs, LSTMs, Transformers) contains FFN components at its core.

The term *feedforward* describes the direction data moves: from input, through one or more hidden layers, to output — always forward, never looping back. Each layer is **fully connected**: every neuron receives input from every neuron in the previous layer. In PyTorch this is `nn.Linear`.

## The three design decisions

| Decision | Guidance |
|----------|----------|
| **How many layers?** | Start with 2–3 hidden layers. More layers capture more complex patterns but are slower to train. |
| **How many neurons per layer?** | 64–512 is a common starting range. Start wider, then add regularisation if the model overfits. |
| **What activation function?** | ReLU for hidden layers. The output layer has no activation — `CrossEntropyLoss` handles Softmax internally. |

## Defining a network in PyTorch

Every network is a Python class inheriting from `nn.Module`. The constructor must accept keyword arguments that **exactly match the keys in `model_config`**, because `ModelPipeline.load()` reconstructs the model by calling `ModelClass(**checkpoint['model_config'])`. This contract was established in Chapter 3 and is followed throughout this book.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# The FFN class
#
# Constructor kwargs must match model_config exactly so ModelPipeline.load()
# can reconstruct the architecture with: ModelClass(**checkpoint['model_config'])
# Same contract as CaliforniaNet in Chapter 3.
# ─────────────────────────────────────────────────────────────────────────────

class FFN(nn.Module):
    """Flexible feedforward network for classification or regression.
    Stacks Linear -> ReLU -> Dropout blocks. Output layer has no activation.
    Constructor kwargs match model_config keys for ModelPipeline compatibility.
    """
    def __init__(self, input_size, hidden_sizes, output_size, dropout_p=0.3):
        super().__init__()
        layers = []
        prev   = input_size
        for h in hidden_sizes:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(p=dropout_p)]
            prev    = h
        layers.append(nn.Linear(prev, output_size))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Compare three architectures at the MNIST scale
print(f"{'Architecture':<30} {'Hidden layers':>14} {'Parameters':>12}")
print('-' * 58)
for label, hidden in [
    ('Shallow  (1 hidden layer)',  [64]),
    ('Medium   (2 hidden layers)', [128, 64]),
    ('Deep     (3 hidden layers)', [256, 128, 64]),
]:
    m = FFN(input_size=784, hidden_sizes=hidden, output_size=10)
    print(f'  {label:<28} {len(hidden):>14} {count_parameters(m):>12,}')

print()
print('input_size=784 (28x28 MNIST pixels), output_size=10 (digits 0-9).')

### 📝 Exercise 4.1 — Design Your Own Architecture

A retail bank wants to classify loan applications into three risk tiers: Low, Medium, High. The application form captures 12 numeric features (income, age, loan amount, existing debts, etc.).

Replace each `None` with your chosen value and run the cell.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 4.1 — Design a network for the loan risk problem
#
# Replace each None with your answer, then run the cell.
#   input_size   : how many features does the application form have?
#   hidden_sizes : choose neuron counts for 2 hidden layers, e.g. [64, 32]
#   output_size  : how many risk tiers?
# ─────────────────────────────────────────────────────────────────────────────

# loan_model = FFN(
#     input_size   = None,          # replace None
#     hidden_sizes = [None, None],  # replace each None
#     output_size  = None,          # replace None
#     dropout_p    = 0.3,
# )
# print(f'Loan risk model — parameters: {count_parameters(loan_model):,}')
# print(loan_model)

# ── Suggested answer ─────────────────────────────────────────────────────────
loan_model = FFN(
    input_size   = 12,
    hidden_sizes = [64, 32],
    output_size  = 3,
    dropout_p    = 0.3,
)
print(f'Loan risk model — parameters: {count_parameters(loan_model):,}')
print(loan_model)

---
# 4.2 Warm-Up: MNIST Digit Recognition

## Why MNIST?

MNIST is 70,000 handwritten digits (0–9), each a 28×28 grayscale image. It is the standard sanity-check dataset in deep learning: well-understood, perfectly clean, and fast to train on CPU. It lets us verify that the `FFN` class and the training helpers work correctly before applying them to messier business data.

MNIST is a **multi-class classification** task. The network receives 784 pixel values (28×28 flattened) and predicts which of the 10 digit classes the image belongs to.

**Loss function: `CrossEntropyLoss`**  
`nn.CrossEntropyLoss` combines Softmax and cross-entropy in one numerically stable step. The FFN output layer therefore has no activation — PyTorch applies Softmax internally inside the loss.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Load MNIST via torchvision — no Kaggle account needed
# ─────────────────────────────────────────────────────────────────────────────

mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # pixel values -> [-1, 1]
])

mnist_train = torchvision.datasets.MNIST('./data', train=True,  download=True, transform=mnist_transform)
mnist_test  = torchvision.datasets.MNIST('./data', train=False, download=True, transform=mnist_transform)

mnist_train_loader = DataLoader(mnist_train, batch_size=256, shuffle=True)
mnist_val_loader   = DataLoader(mnist_test,  batch_size=256, shuffle=False)

print(f'Training samples : {len(mnist_train):,}')
print(f'Test samples     : {len(mnist_test):,}')
print(f'Image shape      : {mnist_train[0][0].shape}  (C, H, W)')
print(f'Classes          : {mnist_train.classes}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 4.1 — Sample MNIST images
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 10, figsize=(14, 3.2))
fig.patch.set_facecolor('white')
for i, ax in enumerate(axes.flat):
    img, label = mnist_train[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(str(label), fontsize=10)
    ax.axis('off')
fig.suptitle('Figure 4.1 — Sample MNIST digits (label above each image)',
             fontsize=11, fontweight='bold', color='#1F3864', y=1.04)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MNIST model — model_config keys match FFN constructor kwargs
# ─────────────────────────────────────────────────────────────────────────────

mnist_config = {
    'input_size'  : 784,
    'hidden_sizes': [256, 128],
    'output_size' : 10,
    'dropout_p'   : 0.3,
}

torch.manual_seed(42)
mnist_model = FFN(**mnist_config).to(device)
mnist_crit  = nn.CrossEntropyLoss()
mnist_opt   = optim.Adam(mnist_model.parameters(), lr=1e-3)

print(mnist_model)
print(f'\nTrainable parameters: {count_parameters(mnist_model):,}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Train MNIST using run_training()
#
# MNIST images are (1, 28, 28) tensors. The FFN expects flat vectors of 784
# values. FlattenLoader wraps the DataLoader and flattens images on the fly,
# so run_training() can be used unchanged.
# ─────────────────────────────────────────────────────────────────────────────

class FlattenLoader:
    """Wraps an image DataLoader and flattens (B, C, H, W) -> (B, C*H*W)."""
    def __init__(self, loader): self.loader = loader
    def __len__(self):          return len(self.loader)
    def __iter__(self):
        for imgs, labels in self.loader:
            yield imgs.view(imgs.size(0), -1), labels
    @property
    def dataset(self):          return self.loader.dataset


flat_train = FlattenLoader(mnist_train_loader)
flat_val   = FlattenLoader(mnist_val_loader)

print('Training MNIST (10 epochs):')
mnist_train_losses, mnist_val_losses = run_training(
    model         = mnist_model,
    train_loader  = flat_train,
    val_loader    = flat_val,
    criterion     = mnist_crit,
    optimiser     = mnist_opt,
    num_epochs    = 10,
    device        = device,
    verbose_every = 1,
)
print('\nTraining complete.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 4.2 — Loss curves and test accuracy
# ─────────────────────────────────────────────────────────────────────────────

mnist_model.eval()
correct = total = 0
with torch.no_grad():
    for imgs, labels in flat_val:
        preds    = mnist_model(imgs.to(device)).argmax(dim=1).cpu()
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
test_acc = 100.0 * correct / total

epochs_range = range(1, 11)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
fig.patch.set_facecolor('white')

ax1.plot(epochs_range, mnist_train_losses, color='#2980b9', lw=2.5, label='Train loss')
ax1.plot(epochs_range, mnist_val_losses,   color='#e74c3c', lw=2.5, linestyle='--', label='Val loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Figure 4.2a — Loss curves', fontweight='bold', color='#1F3864')
ax1.legend()

ax2.plot(epochs_range, mnist_val_losses, color='#27ae60', lw=2.5, marker='o', markersize=5)
ax2.axhline(y=mnist_val_losses[-1], color='#27ae60', lw=1, linestyle=':', alpha=0.6)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Val loss')
ax2.set_title('Figure 4.2b — Validation loss convergence', fontweight='bold', color='#1F3864')

plt.tight_layout()
plt.show()
print(f'Final test accuracy: {test_acc:.2f}%')

## Confusion Matrix

Accuracy tells you how often the model is right overall. The confusion matrix shows *where* it goes wrong. Each row is an actual class; each column is a predicted class. The diagonal holds correct predictions — everything off the diagonal is a mistake. Colour intensity shows how frequently each error occurs.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 4.3 — Confusion matrix on the full test set
# ─────────────────────────────────────────────────────────────────────────────

all_preds, all_labels = [], []
mnist_model.eval()
with torch.no_grad():
    for imgs, labels in flat_val:
        all_preds.extend(mnist_model(imgs.to(device)).argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10), linewidths=0.5, ax=ax)
ax.set_xlabel('Predicted digit', fontsize=12)
ax.set_ylabel('Actual digit', fontsize=12)
ax.set_title('Figure 4.3 — Confusion Matrix: MNIST Test Set\n'
             'Diagonal = correct  |  Off-diagonal = errors  |  Colour = frequency',
             fontsize=12, fontweight='bold', color='#1F3864')
plt.tight_layout()
plt.show()

cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)
print('Top-5 most common errors (actual -> predicted):')
for _ in range(5):
    idx = np.unravel_index(cm_no_diag.argmax(), cm_no_diag.shape)
    print(f'  Digit {idx[0]} predicted as {idx[1]}: {cm_no_diag[idx]} times')
    cm_no_diag[idx] = 0

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 4.4 — Misclassified examples
# ─────────────────────────────────────────────────────────────────────────────

misclassified = [(img, t, p) for img, t, p in
                 zip(mnist_test.data, all_labels, all_preds) if t != p]

fig, axes = plt.subplots(2, 10, figsize=(15, 3.5))
fig.patch.set_facecolor('white')
for i, ax in enumerate(axes.flat):
    if i < len(misclassified):
        img, t, p = misclassified[i]
        ax.imshow(img, cmap='gray')
        ax.set_title(f'T:{t} P:{p}', fontsize=8, color='#c0392b')
    ax.axis('off')
fig.suptitle('Figure 4.4 — Misclassified examples  (T = true label, P = predicted)',
             fontsize=11, fontweight='bold', color='#1F3864', y=1.04)
plt.tight_layout()
plt.show()
print(f'Total misclassified: {len(misclassified):,} of {len(mnist_test):,} '
      f'({100*len(misclassified)/len(mnist_test):.2f}% error rate)')

---
# 4.3 Evaluation Metrics for Classification

## Why accuracy is not enough

MNIST is well-balanced — each digit appears roughly equally — so accuracy is a fair summary. Most real business problems are not balanced. Consider customer churn: if 85% of customers stay and only 15% churn, a model that always predicts "no churn" achieves 85% accuracy without learning anything useful. Accuracy would call this an excellent model. The business would discover otherwise.

Four metrics together give the full picture:

| Metric | Formula | Business meaning |
|--------|---------|------------------|
| **Precision** | TP / (TP + FP) | Of all customers flagged as churners, what fraction actually churned? |
| **Recall** | TP / (TP + FN) | Of all customers who actually churned, what fraction did we catch? |
| **F1 Score** | 2 × (P × R) / (P + R) | Single number that balances precision and recall |
| **ROC-AUC** | Area under the ROC curve | Overall ability to separate classes regardless of threshold |

## The precision–recall trade-off

To catch every churner (high recall) you must flag more customers — including loyal ones (lower precision). The right balance depends on the relative cost of each error type, which we calculate explicitly in Section 4.4.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 4.5 — The four outcomes of a binary classifier + P-R trade-off
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.patch.set_facecolor('white')

ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis('off'); ax.set_facecolor('white')
for x, y, fc, ec, title, sub in [
    (2.5, 4.2, '#d5f5e3', '#27ae60', 'True Positive (TP)',  'Churner correctly identified'),
    (7.0, 4.2, '#fadbd8', '#e74c3c', 'False Positive (FP)', 'Loyal customer flagged'),
    (2.5, 2.5, '#fdebd0', '#e67e22', 'False Negative (FN)', 'Churner missed by model'),
    (7.0, 2.5, '#d4e6f1', '#2980b9', 'True Negative (TN)',  'Loyal customer correctly ignored'),
]:
    ax.add_patch(mpatches.FancyBboxPatch(
        (x-1.75, y-0.6), 3.5, 1.2, boxstyle='round,pad=0.1',
        facecolor=fc, edgecolor=ec, lw=2))
    ax.text(x, y+0.22, title, ha='center', fontsize=9, fontweight='bold')
    ax.text(x, y-0.22, sub,   ha='center', fontsize=8, color='#444')
ax.text(5, 5.7, 'Predicted: Churn',    ha='center', fontsize=9, color='#555', fontstyle='italic')
ax.text(5, 1.5, 'Predicted: No Churn', ha='center', fontsize=9, color='#555', fontstyle='italic')
ax.text(0.4, 4.2, 'Actual:\nChurn',    ha='center', fontsize=9, color='#555', fontstyle='italic', rotation=90)
ax.text(0.4, 2.5, 'Actual:\nNo Churn', ha='center', fontsize=9, color='#555', fontstyle='italic', rotation=90)
ax.set_title('Figure 4.5a — The Four Outcomes', fontsize=10, fontweight='bold', color='#1F3864')

ax2 = axes[1]
t_sim = np.linspace(0.1, 0.95, 100)
ax2.plot(0.98 - 0.85 * t_sim, 0.4 + 0.55 * t_sim, color='#8e44ad', lw=2.5)
for t, lbl, col in [(0.2, 'High recall', '#e74c3c'),
                    (0.5, 'Balanced',    '#f39c12'),
                    (0.8, 'High precision', '#27ae60')]:
    idx = int(t * 99)
    r, p = 0.98 - 0.85*t_sim[idx], 0.4 + 0.55*t_sim[idx]
    ax2.scatter(r, p, s=120, color=col, zorder=5)
    ax2.annotate(lbl, xy=(r, p), xytext=(r-0.05, p+0.07), fontsize=8, color=col, ha='center')
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.set_xlim(0, 1); ax2.set_ylim(0.3, 1.05)
ax2.set_title('Figure 4.5b — Precision-Recall Trade-Off',
              fontsize=10, fontweight='bold', color='#1F3864')
plt.tight_layout()
plt.show()

---
# 4.4 Business Application: Customer Churn Prediction

## The business problem

Customer churn — when a customer stops using a service — is one of the most expensive problems in subscription businesses. Acquiring a new customer typically costs five to ten times more than retaining an existing one. A model that identifies likely churners *before they leave* allows a business to intervene with a targeted offer, a support call, or a pricing adjustment.

The **Telco Customer Churn** dataset from Kaggle contains 7,043 telecommunications customers with demographics, subscribed services, account details, and a churn label (`Yes`/`No`).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Kaggle authentication — same pattern as Chapter 0
# ─────────────────────────────────────────────────────────────────────────────

kaggle_token    = getpass.getpass('Paste your Kaggle API token (KGAT_...): ')
kaggle_username = input('Enter your Kaggle username: ')

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
    json.dump({'username': kaggle_username.strip(), 'key': kaggle_token.strip()}, f)
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)

!kaggle datasets download -d blastchar/telco-customer-churn --unzip -p ./data/churn
print('Download complete.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Load and inspect
# ─────────────────────────────────────────────────────────────────────────────

df = pd.read_csv('./data/churn/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f'Shape: {df.shape}  ({df.shape[0]:,} customers, {df.shape[1]} columns)')
df.head(3)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 4.6 — Class balance and monthly charges distribution
# ─────────────────────────────────────────────────────────────────────────────

churn_counts = df['Churn'].value_counts()
churn_pct    = df['Churn'].value_counts(normalize=True) * 100

print('Class distribution:')
for cls in churn_counts.index:
    print(f'  {cls:<5}  {churn_counts[cls]:>5,}  ({churn_pct[cls]:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor('white')

axes[0].bar(churn_counts.index, churn_counts.values,
            color=['#2980b9', '#e74c3c'], edgecolor='white', linewidth=2)
for i, (cls, cnt) in enumerate(zip(churn_counts.index, churn_counts.values)):
    axes[0].text(i, cnt + 40, f'{cnt:,}\n({churn_pct[cls]:.1f}%)', ha='center', fontsize=10)
axes[0].set_title('Figure 4.6a — Class Imbalance', fontweight='bold', color='#1F3864')
axes[0].set_ylabel('Number of customers')
axes[0].set_ylim(0, churn_counts.max() * 1.2)

colors = {'No': '#2ecc71', 'Yes': '#e74c3c'}  # green for no churn, red for churn

for churn_value, group in df.groupby('Churn'):
    group['MonthlyCharges'].plot(
        kind='density',
        ax=axes[1],
        lw=2.5,
        label='No Churn' if churn_value == 'No' else 'Churn',
        color=colors[churn_value]
    )

axes[1].set_title('Figure 4.6b — Monthly Charges by Churn Status',
                  fontweight='bold', color='#1F3864')
axes[1].set_xlabel('Monthly Charges ($)')
axes[1].legend()
plt.tight_layout()
plt.show()

## Reading Figure 4.6

### Panel (a) — Class Imbalance
This bar chart shows how many customers belong to each class. The blue bar represents customers who did **not** churn; the red bar represents those who did. The count and percentage are labelled above each bar.

The key observation is the **imbalance**: roughly 73% of customers stayed and only 27% churned. This matters because a model that simply predicts "No Churn" for every customer would achieve 73% accuracy without learning anything. Before training, we need to account for this so the model pays appropriate attention to the minority class.

### Panel (b) — Monthly Charges by Churn Status
This is a **density plot** (also called a KDE — Kernel Density Estimate). It shows the distribution of monthly charges separately for churners (red) and non-churners (blue). The y-axis is density — a scaled measure of how concentrated the data is at each charge level — and the area under each curve sums to 1.

The key observation is that churners tend to have **higher monthly charges** — the red curve peaks further to the right than the blue curve. This tells us that `MonthlyCharges` is likely to be a useful predictor: customers paying more per month are more likely to churn, possibly because they feel they are not getting value for money. The model will learn this relationship automatically, but seeing it in the EDA confirms the feature is worth including.

## Handling Class Imbalance

The dataset is roughly 73% non-churners and 27% churners. A naive model could reach 73% accuracy by always predicting "No Churn" — commercially useless. The simplest correction is to pass **class weights** to `CrossEntropyLoss`. The minority class (churn = 1) receives a higher weight, so the model is penalised more for missing a churner than for a false alarm.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Preprocessing — same pipeline pattern as Chapter 2
#
# Steps:
#   1. Fix TotalCharges (stored as string); drop customerID
#   2. Encode target and all categorical columns with LabelEncoder
#   3. Split 70/15/15 BEFORE scaling (prevents data leakage)
#   4. Fit StandardScaler on training data only; transform all three splits
# ─────────────────────────────────────────────────────────────────────────────

df_clean = df.copy()
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce').fillna(0.0)
df_clean.drop(columns=['customerID'], inplace=True)
df_clean['Churn'] = (df_clean['Churn'] == 'Yes').astype(int)

for col in df_clean.select_dtypes(include='object').columns:
    df_clean[col] = LabelEncoder().fit_transform(df_clean[col])

feature_names = df_clean.drop(columns=['Churn']).columns.tolist()
X = df_clean[feature_names].values.astype(np.float32)
y = df_clean['Churn'].values.astype(np.int64)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# Class weights inversely proportional to class frequency
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
class_weights = torch.tensor(
    [len(y_train)/(2*n_neg), len(y_train)/(2*n_pos)], dtype=torch.float32
).to(device)

print(f'Training   : {X_train.shape[0]:,} samples')
print(f'Validation : {X_val.shape[0]:,} samples')
print(f'Test       : {X_test.shape[0]:,} samples')
print(f'Features   : {X_train.shape[1]}')
print(f'Class weights — No Churn: {class_weights[0]:.3f}  |  Churn: {class_weights[1]:.3f}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DataLoaders
# ─────────────────────────────────────────────────────────────────────────────

def make_loader(X, y, batch_size=64, shuffle=False):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                       torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

churn_train_loader = make_loader(X_train, y_train, shuffle=True)
churn_val_loader   = make_loader(X_val,   y_val)
churn_test_loader  = make_loader(X_test,  y_test)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Churn model — model_config keys match FFN constructor kwargs exactly
# ─────────────────────────────────────────────────────────────────────────────

NUM_FEATURES = X_train.shape[1]

churn_config = {
    'input_size'  : NUM_FEATURES,
    'hidden_sizes': [128, 64, 32],
    'output_size' : 2,
    'dropout_p'   : 0.3,
}

torch.manual_seed(42)
churn_model = FFN(**churn_config).to(device)
churn_crit  = nn.CrossEntropyLoss(weight=class_weights)
churn_opt   = optim.Adam(churn_model.parameters(), lr=1e-3)
churn_sched = optim.lr_scheduler.ReduceLROnPlateau(churn_opt, patience=5, factor=0.5)
churn_es    = EarlyStopping(patience=10)

print(churn_model)
print(f'\nTrainable parameters: {count_parameters(churn_model):,}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Train churn model using run_training()
#
# ReduceLROnPlateau is passed via val_loss_scheduler= so run_training()
# calls it correctly as sched.step(val_loss) after every epoch.
# EarlyStopping is passed via early_stopping= and halts the loop automatically.
# ─────────────────────────────────────────────────────────────────────────────

MAX_EPOCHS = 60

print('Training churn model (up to 60 epochs with early stopping):')
churn_train_hist, churn_val_hist = run_training(
    model               = churn_model,
    train_loader        = churn_train_loader,
    val_loader          = churn_val_loader,
    criterion           = churn_crit,
    optimiser           = churn_opt,
    num_epochs          = MAX_EPOCHS,
    device              = device,
    val_loss_scheduler  = churn_sched,   # ReduceLROnPlateau
    early_stopping      = churn_es,
    verbose_every       = 5,
)

print('\nTraining complete. Best weights restored.')

## Full Evaluation

Figure 4.7 shows four panels that together give a complete picture of the trained model.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Collect test set predictions
# ─────────────────────────────────────────────────────────────────────────────

churn_model.eval()
test_preds, test_probs, test_labels = [], [], []

with torch.no_grad():
    for X_batch, y_batch in churn_test_loader:
        logits = churn_model(X_batch.to(device))
        probs  = torch.softmax(logits, dim=1)[:, 1].cpu()
        test_preds.extend(logits.argmax(dim=1).cpu().numpy())
        test_probs.extend(probs.numpy())
        test_labels.extend(y_batch.numpy())

test_preds  = np.array(test_preds)
test_probs  = np.array(test_probs)
test_labels = np.array(test_labels)

print('Classification Report:')
print(classification_report(test_labels, test_preds, target_names=['No Churn', 'Churn']))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 4.7 — Full evaluation dashboard
# ─────────────────────────────────────────────────────────────────────────────

FN_COST = 200   # cost of missing a churner
FP_COST = 20    # cost of a false alarm

sweep_thresholds = np.linspace(0.1, 0.9, 80)
total_costs = []
for t in sweep_thresholds:
    preds_t = (test_probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(test_labels, preds_t, labels=[0,1]).ravel()
    total_costs.append(fn * FN_COST + fp * FP_COST)

best_t_idx = np.argmin(total_costs)
best_t     = sweep_thresholds[best_t_idx]
best_cost  = total_costs[best_t_idx]
fpr, tpr, _ = roc_curve(test_labels, test_probs)
roc_auc     = auc(fpr, tpr)

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.patch.set_facecolor('white')

# (a) Loss curves
ax = axes[0, 0]
ep = range(1, len(churn_train_hist)+1)
ax.plot(ep, churn_train_hist, color='#2980b9', lw=2.5, label='Train loss')
ax.plot(ep, churn_val_hist,   color='#e74c3c', lw=2.5, linestyle='--', label='Val loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('(a) Loss curves', fontweight='bold', color='#1F3864')
ax.legend()

# (b) Confusion matrix
ax = axes[0, 1]
sns.heatmap(confusion_matrix(test_labels, test_preds), annot=True, fmt='d',
            cmap='Blues', xticklabels=['No Churn','Churn'],
            yticklabels=['No Churn','Churn'], linewidths=1, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('(b) Confusion matrix — test set', fontweight='bold', color='#1F3864')

# (c) ROC curve
ax = axes[1, 0]
ax.plot(fpr, tpr, color='#8e44ad', lw=2.5, label=f'AUC = {roc_auc:.3f}')
ax.plot([0,1],[0,1], color='#aaa', lw=1.5, linestyle='--', label='Random')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('(c) ROC curve', fontweight='bold', color='#1F3864')
ax.legend(loc='lower right')

# (d) Cost analysis
ax = axes[1, 1]
ax.plot(sweep_thresholds, total_costs, color='#e67e22', lw=2.5)
ax.axvline(best_t, color='#c0392b', lw=2, linestyle='--')
ax.scatter([best_t], [best_cost], color='#c0392b', s=120, zorder=5)
ax.annotate(
    f'Optimal threshold: {best_t:.2f}\nTotal cost: ${best_cost:,}',
    xy=(best_t, best_cost), xytext=(best_t+0.08, best_cost+500),
    fontsize=9, color='#c0392b',
    arrowprops=dict(arrowstyle='->', color='#c0392b'))
ax.set_xlabel('Decision threshold'); ax.set_ylabel('Total business cost ($)')
ax.set_title(f'(d) Cost analysis  (FN=${FN_COST}, FP=${FP_COST})',
             fontweight='bold', color='#1F3864')

fig.suptitle('Figure 4.7 — Churn Model: Full Evaluation Dashboard',
             fontsize=14, fontweight='bold', color='#1F3864', y=1.01)
plt.tight_layout()
plt.show()
print(f'ROC AUC: {roc_auc:.4f}')
print(f'Business-optimal threshold: {best_t:.2f}  |  Estimated total cost: ${best_cost:,}')

## Reading Figure 4.7 panel by panel

### Panel (a) — Loss curves
The blue line is training loss and the red dashed line is validation loss. Both should fall together and level off at a similar value. If training loss keeps falling while validation loss rises, the model is overfitting — memorising training examples rather than learning general patterns. Early stopping monitors this and halts training at the epoch where validation loss was lowest, then restores those best weights. What you evaluate here is not the final epoch but the best epoch seen during the entire run.

### Panel (b) — Confusion matrix
Read the matrix row by row. The **top row** covers customers who actually did *not* churn: the top-left cell (True Negatives) shows how many were correctly left alone, and the top-right cell (False Positives) shows how many were incorrectly flagged as churners. The **bottom row** covers actual churners: the bottom-right cell (True Positives) shows how many were correctly caught, and the bottom-left cell (False Negatives) shows how many were missed entirely. For a churn use case, the **bottom-left cell is the most expensive** — those are churners the business had no chance to retain.

### Panel (c) — ROC curve
The ROC curve plots **True Positive Rate** (recall — the fraction of actual churners we catch) against **False Positive Rate** (the fraction of loyal customers we incorrectly flag) at every possible decision threshold. A perfect classifier hugs the top-left corner; a random classifier follows the diagonal. The **AUC (Area Under the Curve)** condenses this into one number: 1.0 is perfect, 0.5 is random. An AUC above 0.80 is generally considered good for a business classification problem. The important property of the ROC curve is that it is **threshold-independent** — it measures the model's inherent ability to separate the two classes, regardless of where you ultimately draw the decision line.

### Panel (d) — Cost analysis
The ROC curve tells you the model's *capability*; the cost curve tells you *where to operate*. Each point on the x-axis is a candidate decision threshold. For each threshold we count the False Negatives (missed churners, costing $200 each) and False Positives (false alarms, costing $20 each), and sum them into a total business cost. The **minimum of this curve is the threshold that minimises total cost** — not the threshold that maximises accuracy or F1. Notice how the optimal threshold depends entirely on the relative cost values. Exercise 4.4 lets you change those costs and observe the threshold shift.

### 📝 Exercise 4.4 — Adjust the Cost Assumptions

The costs above are illustrative. Change `FN_COST_EX` and `FP_COST_EX` to your own values and observe how the optimal threshold shifts.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 4.4 — Your own cost assumptions
#
# Replace each None with a dollar value, then run the cell.
#
# Scenario: high-touch retention team.
#   Each retention call costs $80  (FP_COST_EX = 80)
#   Each lost customer is worth $500 in lifetime value (FN_COST_EX = 500)
#
# Question: does the optimal threshold go up or down compared to the original?
# What does that tell you about how aggressively the model should flag customers?
# ─────────────────────────────────────────────────────────────────────────────

FN_COST_EX = None   # replace None
FP_COST_EX = None   # replace None

# Guard: run the analysis only once both values are set
if FN_COST_EX is None or FP_COST_EX is None:
    print('Replace FN_COST_EX and FP_COST_EX with dollar values above, then re-run.')
else:
    total_costs_ex = []
    for t in sweep_thresholds:
        preds_t = (test_probs >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(test_labels, preds_t, labels=[0, 1]).ravel()
        total_costs_ex.append(fn * FN_COST_EX + fp * FP_COST_EX)

    best_t_ex    = sweep_thresholds[np.argmin(total_costs_ex)]
    best_cost_ex = min(total_costs_ex)

    print(f'Your cost assumptions : FN=${FN_COST_EX}, FP=${FP_COST_EX}')
    print(f'Optimal threshold     : {best_t_ex:.2f}  (original was {best_t:.2f})')
    print(f'Estimated total cost  : ${best_cost_ex:,}')
    print()
    direction = 'lower' if best_t_ex < best_t else 'higher'
    print(f'Threshold moved {direction}. A lower threshold means the model flags'
          f' more customers to avoid missing valuable churners.')

# ── Suggested answer ─────────────────────────────────────────────────────────
# FN_COST_EX = 500
# FP_COST_EX = 80
# The higher FN cost drives the optimal threshold DOWN — the model flags more
# customers (accepting more false alarms) to avoid losing valuable churners.

---
# 4.5 Pipeline: Saving and Deploying the Churn Model

## The ModelPipeline — same class as Chapter 3

In Chapter 3 we built `ModelPipeline` — a production-ready wrapper that bundles everything a trained model needs outside the training notebook. We use the **exact same class here**, with the churn model configured inside it. The only addition is that `predict()` accepts a `threshold` parameter and returns a labelled DataFrame — appropriate for a classification use case.

What `ModelPipeline` saves into a single `.pth` file:

| Component | How it is saved | Why |
|-----------|----------------|-----|
| **Model class** | `dill.dumps(model_class)` | Reconstructed on any machine without training code |
| **Architecture config** | Plain dict (`model_config`) | `ModelClass(**config)` rebuilds the architecture on load |
| **Weights** | `state_dict()` | Pure data — no class dependency |
| **Preprocessor** | Fitted `StandardScaler` | Applied identically to new data at inference time |
| **Feature names** | List of column names | Column-level validation on incoming DataFrames |
| **Feature ranges** | `{col: (min, max)}` with 10% buffer | Catches obviously bad data before it silently corrupts predictions |
| **Training history** | `{train_loss, val_loss}` lists | Preserved and extended on every `retrain()` call |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ModelPipeline
#
# Direct copy from Chapter 3 with one extension:
#   predict() accepts a threshold parameter and returns a labelled DataFrame.
# All other methods (save, load, validate_input, retrain) are identical.
# ─────────────────────────────────────────────────────────────────────────────

import dill

class ModelPipeline:
    """Bundles a trained PyTorch model with its preprocessor and metadata.
    Provides: save | load | validate_input | predict | retrain.
    Identical contract to the ModelPipeline introduced in Chapter 3.
    """

    def __init__(self, model, model_config, preprocessor=None,
                 feature_names=None, feature_ranges=None, model_class=None):
        self.model            = model
        self.model_config     = model_config
        self.preprocessor     = preprocessor
        self.feature_names    = feature_names or []
        self.feature_ranges   = feature_ranges or {}
        self._model_class     = model_class
        self.model_class      = model_class.__name__ if model_class else type(model).__name__
        self.training_history = {'train_loss': [], 'val_loss': []}

    def save(self, path):
        """Save the complete pipeline to a single .pth file."""
        cls_obj     = self._model_class if self._model_class is not None else type(self.model)
        class_bytes = dill.dumps(cls_obj)
        checkpoint  = {
            'model_class'       : self.model_class,
            'model_class_bytes' : class_bytes,
            'model_config'      : self.model_config,
            'state_dict'        : self.model.state_dict(),
            'preprocessor'      : self.preprocessor,
            'feature_names'     : self.feature_names,
            'feature_ranges'    : self.feature_ranges,
            'training_history'  : self.training_history,
            'pytorch_version'   : torch.__version__,
            'saved_at'          : datetime.datetime.now().isoformat(),
        }
        torch.save(checkpoint, path)
        print(f'Pipeline saved -> {path}')
        print(f'  model_class : {self.model_class}')
        print(f'  features    : {len(self.feature_names)}')
        print(f'  history     : {len(self.training_history["val_loss"])} epochs recorded')
        print(f'  saved_at    : {checkpoint["saved_at"]}')

    @classmethod
    def load(cls, path, device=None):
        """Load a saved pipeline. No training code needed."""
        device     = device or torch.device('cpu')
        checkpoint = torch.load(path, map_location=device, weights_only=False)
        model_class = dill.loads(checkpoint['model_class_bytes'])
        # Filter to only the keys the FFN constructor accepts
        ffn_keys = {'input_size', 'hidden_sizes', 'output_size', 'dropout_p'}
        ffn_cfg  = {k: v for k, v in checkpoint['model_config'].items() if k in ffn_keys}
        model    = model_class(**ffn_cfg)
        model.load_state_dict(checkpoint['state_dict'])
        model.to(device)
        model.eval()
        pipeline = cls(
            model          = model,
            model_config   = checkpoint['model_config'],
            preprocessor   = checkpoint.get('preprocessor'),
            feature_names  = checkpoint.get('feature_names', []),
            feature_ranges = checkpoint.get('feature_ranges', {}),
            model_class    = model_class,
        )
        pipeline.training_history = checkpoint.get(
            'training_history', {'train_loss': [], 'val_loss': []})
        print(f'Pipeline loaded <- {path}')
        print(f'  model_class    : {checkpoint.get("model_class")}  (reconstructed from dill)')
        print(f'  history        : {len(pipeline.training_history["val_loss"])} epochs recorded')
        print(f'  saved_at       : {checkpoint.get("saved_at")}')
        print(f'  pytorch_version: {checkpoint.get("pytorch_version")}')
        return pipeline

    def validate_input(self, X):
        """Check column names, nulls, and value ranges. Identical to Chapter 3."""
        if isinstance(X, pd.DataFrame):
            missing = set(self.feature_names) - set(X.columns)
            if missing: raise ValueError(f'Missing columns: {sorted(missing)}')
            extra = set(X.columns) - set(self.feature_names)
            if extra: raise ValueError(f'Unexpected extra columns: {sorted(extra)}')
            null_cols = [c for c in self.feature_names if X[c].isnull().any()]
            if null_cols: raise ValueError(f'Null values in columns: {null_cols}')
            for col, (lo, hi) in self.feature_ranges.items():
                if col in X.columns:
                    if X[col].min() < lo or X[col].max() > hi:
                        raise ValueError(
                            f'Column "{col}" out of expected range [{lo:.3f}, {hi:.3f}]')
        else:
            if X.ndim != 2: raise ValueError(f'Expected 2-D array, got shape {X.shape}')
            if self.feature_names and X.shape[1] != len(self.feature_names):
                raise ValueError(f'Expected {len(self.feature_names)} features, got {X.shape[1]}')
            nan_cols = np.where(np.isnan(X).any(axis=0))[0].tolist()
            if nan_cols:
                names = [self.feature_names[i] for i in nan_cols] if self.feature_names else nan_cols
                raise ValueError(f'NaN values in columns: {names}')
            if self.feature_names and self.feature_ranges:
                for i, name in enumerate(self.feature_names):
                    if name in self.feature_ranges:
                        lo, hi = self.feature_ranges[name]
                        if float(X[:, i].min()) < lo or float(X[:, i].max()) > hi:
                            raise ValueError(
                                f'Column "{name}" out of expected range [{lo:.3f}, {hi:.3f}]')
        return True

    def predict(self, X, device=None, threshold=0.5):
        """Validate, preprocess, and predict. Returns a labelled DataFrame."""
        device = device or torch.device('cpu')
        self.validate_input(X)
        if isinstance(X, pd.DataFrame):
            X = X[self.feature_names].to_numpy()
        X = self.preprocessor.transform(X.astype(np.float32))
        self.model.eval()
        with torch.no_grad():
            probs = torch.softmax(
                self.model(torch.tensor(X, dtype=torch.float32).to(device)), dim=1
            )[:, 1].cpu().numpy()
        class_labels = self.model_config.get('class_labels', {0: '0', 1: '1'})
        return pd.DataFrame({
            'churn_probability': np.round(probs, 4),
            'predicted_label'  : [class_labels[int(p >= threshold)] for p in probs],
        })

    def retrain(self, train_loader, val_loader, criterion, optimiser,
                num_epochs, device=None, patience=None):
        """Continue training. Appends to training_history. Identical to Chapter 3."""
        device = device or torch.device('cpu')
        es     = EarlyStopping(patience=patience) if patience else None
        for epoch in range(num_epochs):
            tl = train_one_epoch(self.model, train_loader, criterion, optimiser, device)
            vl = evaluate(self.model, val_loader, criterion, device)
            self.training_history['train_loss'].append(tl)
            self.training_history['val_loss'].append(vl)
            if epoch % 5 == 0:
                print(f'  Epoch {epoch:3d}: train={tl:.4f}  val={vl:.4f}')
            if es and es.step(vl, self.model):
                print(f'  Early stopping at epoch {epoch}  (best val: {es.best_loss:.4f})')
                es.restore_best(self.model)
                break


print('ModelPipeline defined.')
print('Methods: save | load | validate_input | predict | retrain')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Wrap trained model in ModelPipeline and save
#
# Feature ranges: computed from the FULL raw dataset (not just X_train),
# with a 10% buffer — identical to Chapter 3 §3.7c.
# Using the full dataset avoids falsely rejecting legitimate production samples
# that fall slightly outside the training split's observed extremes.
# ─────────────────────────────────────────────────────────────────────────────

X_full_raw = df_clean[feature_names].values.astype(np.float32)
feature_ranges = {}
for i, name in enumerate(feature_names):
    obs_min = float(X_full_raw[:, i].min())
    obs_max = float(X_full_raw[:, i].max())
    span    = obs_max - obs_min
    feature_ranges[name] = (obs_min - span*0.10, obs_max + span*0.10)

# class_labels stored inside model_config — saved and restored automatically
churn_model_config = {
    **churn_config,
    'class_labels': {0: 'No Churn', 1: 'Churn'},
}

pipeline = ModelPipeline(
    model          = churn_model,
    model_config   = churn_model_config,
    preprocessor   = scaler,
    feature_names  = feature_names,
    feature_ranges = feature_ranges,
    model_class    = FFN,
)
pipeline.training_history = {'train_loss': churn_train_hist, 'val_loss': churn_val_hist}

pipeline.save('churn_model_v1.pth')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Load in a clean context and score new customers
# ─────────────────────────────────────────────────────────────────────────────

del pipeline   # simulate a fresh session

loaded = ModelPipeline.load('churn_model_v1.pth')
print(f'\nTraining history restored: {len(loaded.training_history["val_loss"])} epochs')

# predict() expects raw unscaled data — it applies the scaler internally
new_customers_raw = pd.DataFrame(
    scaler.inverse_transform(X_test[:5]), columns=feature_names
)

results = loaded.predict(new_customers_raw, threshold=best_t)
results['actual'] = ['Churn' if l == 1 else 'No Churn' for l in y_test[:5]]

print(f'\nPredictions (threshold = {best_t:.2f}):')
print(results.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Input validation — three failure cases, matching Chapter 3 §3.7f
# ─────────────────────────────────────────────────────────────────────────────

print('Case 1 — missing required column:')
try:
    loaded.predict(new_customers_raw.drop(columns=[feature_names[0]]))
except ValueError as e:
    print(f'  Caught: {e}')

print('\nCase 2 — null value in a column:')
null_df = new_customers_raw.copy()
null_df.iloc[0, 1] = np.nan
try:
    loaded.predict(null_df)
except ValueError as e:
    print(f'  Caught: {e}')

print('\nCase 3 — values wildly outside training range:')
oor_df = new_customers_raw.copy()
oor_df[feature_names[0]] = 999999.0
try:
    loaded.predict(oor_df)
except ValueError as e:
    print(f'  Caught: {e}')

print('\nCase 4 — valid input (all checks pass):')
print(f'  validate_input returned: {loaded.validate_input(new_customers_raw)}  (True)')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Retrain on a new month of data
#
# Uses the same retrain() signature as Chapter 3:
#   retrain(train_loader, val_loader, criterion, optimiser, num_epochs, device)
#
# Training history accumulates — the full run is visible after saving v2.
# ─────────────────────────────────────────────────────────────────────────────

retrain_opt = optim.Adam(loaded.model.parameters(), lr=5e-4)

print('Retraining on new data (10 additional epochs):')
loaded.retrain(
    train_loader = churn_train_loader,
    val_loader   = churn_val_loader,
    criterion    = nn.CrossEntropyLoss(),
    optimiser    = retrain_opt,
    num_epochs   = 10,
    device       = device,
)

loaded.save('churn_model_v2.pth')
print(f'\nTotal epochs in history: {len(loaded.training_history["val_loss"])}')
print('Version 1 preserved for rollback if needed.')

---
## Chapter 4 Summary

| Concept | Key takeaway |
|---------|-------------|
| **FFN architecture** | `nn.Module` with `Linear -> ReLU -> Dropout` blocks; constructor kwargs must match `model_config` |
| **Training helpers** | `train_one_epoch`, `evaluate`, `run_training`, `EarlyStopping` from Chapter 3 — reused unchanged |
| **MNIST** | 10-class classifier with `CrossEntropyLoss`; accuracy is reliable when classes are balanced |
| **Confusion matrix** | Reveals *where* the model fails — off-diagonal cells show the most common error pairs |
| **Imbalanced classes** | `CrossEntropyLoss(weight=...)` penalises the minority class more; sufficient for moderate imbalance |
| **Precision / Recall / F1** | Three complementary metrics that expose trade-offs hidden by accuracy |
| **ROC-AUC** | Threshold-independent separability measure; above 0.80 is generally good for business problems |
| **Cost analysis** | Translating TP/FP/FN counts into dollar costs reveals the operationally optimal threshold |
| **ModelPipeline** | Same class as Chapter 3 — saves model class (dill), weights, scaler, feature ranges, and training history |

---

## What Comes Next

Chapter 5 introduces **Convolutional Neural Networks (CNNs)** — the architecture built for image data. Where an FFN treats every pixel as an independent input, a CNN learns to recognise local patterns (edges, textures, shapes) regardless of where they appear. The business application is manufacturing defect detection: classifying product images as conforming or defective from a Kaggle dataset of casting process images.

---
*Deep Learning for Business Analytics: From Basics to Large Language Models*  
*Dr. M. Ramasubramaniam &amp; Mr. Daniel Peter*